

<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería y Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Laboratorio de Procesamiento de Datos</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Comprension de los Datos</h3>
    </div>
</div>



**En esta unidad vamos a tratar los siguientes puntos:**

1. Distinguir variables cualitativas, cuantitativas, binarias, nominales y ordinales.
2. Reconocer la diferencia entre el tipo almacenado (`dtype`) y la escala de medición.
3. Diagnosticar variables en un conjunto de datos real con `pandas`.
4. Identificar valores faltantes y elegir una estrategia con criterio.



# 1. Clasificación de Tipos de Datos
## 1.1 ¿Qué es un dato y por qué importa conocer su tipo?

Un **dato** es una representación de una característica de una entidad, observación o evento. En un dataframe, cada columna representa una variable y cada fila una observación. La taxonomía ayuda a responder preguntas prácticas:

- ¿Tiene sentido calcular un promedio?
- ¿Existe un orden entre las categorías?
- ¿Qué codificación necesita el modelo?
- ¿Qué valores representan ausencia, desconocido o una categoría real?

**Dos perspectivas que conviene separar**

| Perspectiva | Pregunta | Ejemplo |
|---|---|---|
| **Tipo almacenado** | ¿Cómo está guardado en Python? | `int64`, `float64`, `object`, `bool` |
| **Significado estadístico** | ¿Qué operaciones representan la realidad? | nominal, ordinal, discreta, continua |

Una columna guardada como número no necesariamente es cuantitativa. Por ejemplo, `1 = rojo`, `2 = azul` sigue siendo **nominal**: los números son etiquetas y no cantidades.

### Escalas y tipos de variable

| Tipo | Característica | Ejemplos | Operaciones razonables |
|---|---|---|---|
| **Binaria** | Dos estados | `yes/no`, `0/1`, presencia/ausencia | proporción, conteo, tasa |
| **Nominal** | Categorías sin orden | trabajo, país, color | frecuencia, moda |
| **Ordinal** | Categorías con orden, sin distancias necesariamente iguales | primaria/secundaria/terciaria, bajo/medio/alto | comparación de orden, mediana |
| **Cuantitativa discreta** | Conteos enteros | número de contactos, hijos, compras | suma, promedio, dispersión |
| **Cuantitativa continua** | Mediciones en una escala | peso, duración, temperatura | operaciones aritméticas y dispersión |


**Clasificar una variable no es adivinar: es seguir una secuencia de preguntas que el código puede responder.**

Esta secuencia es exactamente la que aplicaremos con el archivo `bank.csv`: 

- primero un diagnóstico general (`info`, `nunique`), después separación por tipo almacenado (`select_dtypes`) y finalmente el significado de cada columna (binaria, nominal, ordinal).
- Al final del cuaderno convertiremos esta secuencia en una **función reutilizable** que genera un reporte de calidad y clasificación para cualquier `DataFrame`.

| Paso | Pregunta | Código en `pandas` | Qué nos dice |
|---|---|---|---|
| 1 | ¿Cómo lo guardó pandas? | `df.dtypes`, `df.info()` | Tipo almacenado (`int64`, `float64`, `object`, `bool`, `category`) |
| 2 | ¿Cuántas categorías distintas tiene? | `df[col].nunique()` | Pista de si es binaria (2), categórica (pocas) o posiblemente continua (muchas) |
| 3 | ¿Cuáles son esos valores? | `df[col].unique()`, `df[col].value_counts()` | Si son etiquetas de texto, códigos numéricos disfrazados o una escala real |
4 | ¿Existe un orden lógico entre las categorías? | Revisión manual + `pd.CategoricalDtype(ordered=True)` | Nominal (sin orden) vs. ordinal (con orden) |
| 5 | Si es numérica, ¿son conteos o mediciones continuas? | `df[col].agg(['min','max','nunique'])` | Discreta (enteros, rango acotado) vs. continua (decimales, muchos valores) |
| 6 | ¿Hay valores faltantes que puedan sesgar la clasificación? | `df[col].isna().sum()` | Evita confundir una categoría real con datos ausentes |



Referencias: 
- [pandas `select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html)
- [pandas `CategoricalDtype`](https://pandas.pydata.org/docs/user_guide/categorical.html)
- [scikit-learn: encoding categorical features](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)


###  Caso de estudio: campañas de marketing bancario

Trabajaremos con `bank.csv`. Cada fila representa un contacto de una campaña telefónica y `y` indica si la persona contrató el producto ofrecido.

Antes de ejecutar el código, tratemos de deducir la clasificación de estas variables:

- `age`, `balance`, `duration`
- `job`, `marital`, `education`, `month`
- `default`, `housing`, `loan`, `y`

La predicción importa: la clasificación que hagas guiará la exploración y la transformación posterior.

In [ ]:
import pandas as pd

In [ ]:
df_bank = pd.read_csv('Data/bank.csv')

In [ ]:
df_bank.head()

In [ ]:
df_bank.tail()

In [ ]:
df_bank.shape

In [ ]:
df_bank.columns

### Lectura rápida 

`info()` permite revisar tamaño, tipos almacenados y valores no nulos. Esta salida es un diagnóstico inicial, no una clasificación completa: una variable `object` puede contener categorías, texto libre o fechas mal interpretadas.

En la siguiente tabla verificaremos qué tipo de dato observa `pandas` en cada columna, cuántos valores distintos tiene (`nunique`) y cuántos faltantes hay. Estas tres columnas ya permiten anticipar la clasificación:

- **`valores_unicos` alto y `tipo_python` numérico** ---> candidata a **cuantitativa** (falta decidir si discreta o continua).

- **`valores_unicos` muy bajo (2)** ---> candidata a variable **binaria**.

- **`valores_unicos` bajo y `tipo_python == object`** ---> candidata a **categórica** (falta decidir si nominal u ordinal).

In [ ]:
df_bank.info()

In [ ]:
schema_bank = pd.DataFrame({
    'tipo_python': df_bank.dtypes.astype(str),
    'valores_unicos': df_bank.nunique(),
})
schema_bank

## 1.2 Variables cualitativas y cuantitativas

`select_dtypes()` clasifica por el tipo almacenado, lo que es útil como primer filtro: separa en un solo paso las columnas numéricas (`include='number'`) de las que pandas interpreta como texto u objeto (`include='object'`). 

Esto es rápido, pero **no** es la clasificación final: después debemos revisar el significado de cada columna y documentar excepciones (por ejemplo, un código numérico que en realidad es una etiqueta nominal).

In [ ]:
df_bank.head(2)

In [ ]:
# Obteniendo datos cuantitativos
df_bank_cuantitativos = df_bank.select_dtypes(include='number')
df_bank_cuantitativos.head()

In [ ]:
df_bank_cuantitativos.columns

## 1.3 Cuantitativas discretas y continuas

La separación `number` / `object` no basta. En este dataset, `age`, `balance` y `duration` son medidas cuantitativas; `campaign`, `pdays` y `previous` son conteos o códigos numéricos cuyo significado merece una revisión adicional.

Una heurística práctica para distinguirlas con código:

Podemos observar sus rangos y valores únicos antes de elegir una transformación:

- Si el `dtype` es entero (`int64`) **y** el número de valores únicos es pequeño en relación al total de filas, suele tratarse de un **conteo discreto** (p. ej. `campaign`, `previous`).
- 
- Si el `dtype` es decimal (`float64`) o el número de valores únicos es muy alto, suele tratarse de una **medición continua** (p. ej. `balance`, `duration`).

In [ ]:
df_bank_cuantitativos.head()

In [ ]:
columnas_revisar = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
df_bank[columnas_revisar].agg(['min', 'max', 'nunique']).T

In [ ]:
#Obteniendo datos cualitativos
df_bank_categoricos = df_bank.select_dtypes(exclude='number')
df_bank_categoricos.head()

In [ ]:
df_bank_categoricos.columns

## 1.4 Variables binarias

Las columnas `default`, `housing`, `loan` y `y` tienen dos categorías. La codificación `yes/no` conserva legibilidad.

>En código, la señal más directa de una variable **binaria** es `nunique() == 2`, sin importar si el `dtype` es `object`, `int64` o `bool`. 

Por eso conviene revisar `nunique()` para **todas** las columnas, no solo las categóricas.

In [ ]:
#Otra alternativa para obtener datos categóricos
df_bank.select_dtypes(exclude='number')

In [ ]:
df_bank_categoricos['education'].nunique(), df_bank_categoricos['education'].unique()

In [ ]:
df_bank_categoricos['education'].value_counts()

In [ ]:
#Guardamos las columnas originales de df_bank
columns_bank = df_bank.columns
columns_bank

In [ ]:
#Guardar las columnas de las variables cuantitivas
columns_cuantitativas = df_bank_cuantitativos.columns
columns_cuantitativas

In [ ]:
#Guardar las columnas de las variables categóricas
columns_categoricas = df_bank_categoricos.columns
columns_categoricas

In [ ]:
# Categóricos (Multiestado o cualitativos) --->  (nominales, ordinales)
df_bank_categoricos.head()

In [ ]:
# Valores únicos y categorías de la columna default
df_bank_categoricos['default'].unique(), df_bank_categoricos['default'].nunique()

In [ ]:
# Lista de columnas y sus categorías
for col in columns_categoricas:
    print(f'>> La columna {col} tiene {df_bank_categoricos[col].nunique()} categorías y son:\n {df_bank_categoricos[col].unique()} \n')

## 1.5 Variables Categoricas ordinales

**Las variables categóricas ordinales son variables con categorías con orden, sin distancias necesariamente iguales**

Siguiendo nuestro ejemplo, `education` representa niveles educativos con un orden razonable. Aun así, la distancia entre niveles no tiene por qué ser igual: pasar de `primary` a `secondary` no equivale necesariamente a pasar de `secondary` a `tertiary`.

`month` es otra variable categórica ordinal, esta tiene una secuencia temporal, pero es una variable cíclica: sin embargo, diciembre y enero están próximos en el calendario. 

A diferencia de obtener una variable binaria o numérica, **el código no puede detectar el orden por sí solo**: 

- `nunique()` y `unique()` muestran cuántas categorías hay y cuáles son, pero decidir que `primary < secondary < tertiary` requiere conocimiento del dominio. Por eso necesitamos declarar explícitamente la lista de variable `ordinales_cat` y el orden con `pd.CategoricalDtype(ordered=True)`.

In [ ]:
# `education` tiene un orden natural; `month` es temporal y conviene tratarlo aparte.
ordinales_cat = ['education']
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna != 'y'
]
binarias_cat = ['default', 'housing', 'loan', 'y']


In [ ]:
binarias_cat

In [ ]:
# dataframes con datos ordinales categóricos
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat]
df_bank_categoricos_ord.head()

In [ ]:
# Declarar el orden evita que una transformación posterior lo invente o lo pierda.
education_order = ['primary', 'secondary', 'tertiary']
education_dtype = pd.api.types.CategoricalDtype(
    categories=education_order,
    ordered=True
)
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat].copy()
df_bank_categoricos_ord['education'] = df_bank_categoricos_ord['education'].astype(education_dtype)
df_bank_categoricos_ord['education'].dtype

## 1.7 Categóricas Nominales

Una vez descartadas las binarias y las ordinales, lo que queda entre las columnas de tipo `object` son **nominales**: categorías sin orden natural (`job`, `marital`, `contact`, `poutcome`, etc.).

- Aquí el criterio es por descarte: si no tiene exactamente dos valores y no le asignamos un orden explícito, es nominal.

In [ ]:
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna not in binarias_cat
]
nominales_cat

In [ ]:
nominales_cat

In [ ]:
# Base de datos con datos nominales categóricos
df_bank_categoricos_nom = df_bank_categoricos[nominales_cat]
df_bank_categoricos_nom


# 2. Valores faltantes: identificar antes de imputar

Un **valor faltante (o missing value)** es la ausencia de un dato o información en una celda específica para una variable dentro de un conjunto de datos

Un valor faltante no siempre significa lo mismo: 

- Puede ser una medición no realizada, una respuesta omitida o un valor que no aplica.

Antes de decidir, necesitamos cuantificar el problema y consultar el significado de la variable.

En este cuaderno solo hacemos una introducción. La comparación detallada de métodos de imputación continúa en el módulo de **Tratamiento de datos faltantes**.

In [ ]:
#Cargamos un nuevo ejemplo dataset de peliculas
df_movie = pd.read_csv('Data/movie_metadata.csv')
df_movie.head()

In [ ]:
#Obtenemos la información general del dataframe
df_movie.info()

In [ ]:
# missing_summary
missing_summary = df_movie.isna().agg(['sum', 'mean']).T.rename(columns = {'sum':'n_faltantes', 'mean':'proporcion_faltantes'})
missing_summary

In [ ]:
df_movie['color']

### Manejando datos Faltantes (intro)

In [ ]:
# Opción 1: eliminar filas completas solo cuando la pérdida sea aceptable.
df_movie_clean = df_movie.dropna()
df_movie_clean

In [ ]:
df_movie_clean.shape

In [ ]:
#verificar que no se tengan valores faltantes
df_movie_clean.info()

## tratar datos faltantes en columnas numéricas

In [ ]:
# Para columnas numéricas
df_movie['duration']

In [ ]:
#promedio
df_movie['duration'].fillna(df_movie['duration'].mean())
df_movie

In [ ]:
# Opción 2: imputar una columna numérica con la mediana.
# La mediana suele ser más resistente a valores extremos que el promedio.
df_movie['duration'].median()

## Estrategias iniciales para valores faltantes

La estrategia depende del tipo de variable y del contexto. 

- Eliminar filas solo si la pérdida es pequeña y no introduce sesgo
- Imputar con estadísticas calculadas en el conjunto de entrenamiento cuando preparemos un modelo de ML.

Para datos categorícos, `Unknown` debe distinguirse de una categoría real. 

En un proyecto de ciencia de datos, la recomendación es documentar cuántos valores fueron imputados y por qué.

In [ ]:
df_movie['color'].unique()

## 3. Reporte de calidad de datos: Automatizar la clasificación

Repetir manualmente los pasos anteriores para cada columna es lento y propenso a errores en datasets con muchas variables. 

Podemos convertir la secuencia de preguntas de la sección 1.1.1 en una **función reutilizable** que recorra un `DataFrame` y devuelva otro `DataFrame` con:

- El tipo almacenado (`dtype`) y el tipo sugerido (binaria, categórica nominal, categórica ordinal, cuantitativa discreta o continua).
- Indicadores de calidad: número de valores únicos, número y porcentaje de valores faltantes.
- Un ejemplo de los valores observados, útil para auditar rápidamente el resultado.

La función usa heurísticas (número de valores únicos, `dtype`) para proponer un tipo, pero deja como **parámetros** las columnas que el analista ya sabe que son binarias u ordinales, porque esa información depende del significado del negocio y no puede inferirse solo del código.


In [ ]:
def reporte_calidad_datos(df, ordinales=None, binarias=None, max_categorias_discretas=20):
    """Clasifica cada columna de df y arma un reporte de calidad de datos.

    Parameters
    ----------
    df : pd.DataFrame
        Datos a diagnosticar.
    ordinales : list[str], opcional
        Columnas que el analista ya identificó como categóricas ordinales.
    binarias : list[str], opcional
        Columnas que el analista ya identificó como binarias (además de las
        que se detectan automáticamente por tener 2 valores únicos).
    max_categorias_discretas : int
        Umbral de valores únicos para distinguir una cuantitativa discreta
        (pocos valores enteros distintos) de una continua.

    Returns
    -------
    pd.DataFrame
        Una fila por variable con su clasificación y métricas de calidad.
    """
    
    return None


### Aplicando el reporte a `bank.csv`

Le pasamos las columnas que ya sabemos que son ordinales (`education`) y binarias (`default`, `housing`, `loan`, `y`); el resto se clasifica automáticamente.


### Aplicando el reporte a `movie_metadata.csv`

Sin pasar `ordinales` ni `binarias`, la función solo puede basarse en heurísticas: detecta binarias por conteo de categorías y separa numéricas de nominales, pero no puede saber si alguna columna tiene un orden. Comparar este resultado con lo que ya sabes del dataset ayuda a detectar columnas que requieren revisión manual.


## Práctica de Laboratorio: Clasificación de tipos de Datos

Responde y justifica tus decisiones. No existe una única respuesta correcta si explicas el criterio.

1. Clasifica `job`, `education`, `month`, `campaign` y `y` según su significado estadístico.
2. ¿Por qué no sería correcto calcular el promedio de `job` aunque se codifique con números?
3. ¿Qué problema puede aparecer si se codifica `month` como `1, 2, ..., 12` y se usa esa columna directamente en un modelo?
4. Calcula la proporción de personas con `y == 'yes'` y compárala por nivel de `education`.
5. Elige entre eliminar o imputar los faltantes de `movie_metadata.csv`. Reporta cuántas filas o valores afecta tu decisión.
6. Ejecuta `reporte_calidad_datos(df_movie)` y revisa el resultado: ¿hay alguna columna cuyo `tipo_sugerido` te parezca incorrecto? Explica por qué la heurística falla en ese caso.

### Reto

Modifica `reporte_calidad_datos` para que agregue una columna `advertencia` que marque con `True` las variables con más de 30% de valores faltantes.